# IRI Compute + Filesystem Smoke Test

This notebook runs an end-to-end smoke test across two resources:

1. **Compute**: submit a job that writes a timestamped log file containing date/time/hostname/user and a full environment dump.
2. **Filesystem**: list the job directory (pretty table) and download the log file via async FS tasks, then **base64-decode** and print it.

## Before you run
Use login-globus or login-esnet notebook to receive a token. 

## 1) Imports & helper functions
Utilities used throughout the notebook.


In [ ]:
import os
import sys
import time
import base64
import json
import datetime as dt
from pathlib import Path
from typing import Any, Dict, List, Optional

import requests
from dotenv import load_dotenv

load_dotenv()

def die(msg: str) -> None:
    print(f"\nERROR: {msg}")
    raise SystemExit(1)


def print_table(headers: List[str], rows: List[List[Any]]) -> None:
    widths = [len(h) for h in headers]
    for row in rows:
        for i, cell in enumerate(row):
            widths[i] = max(widths[i], len(str(cell)))
    fmt = " | ".join(f"{{:<{w}}}" for w in widths)
    sep = "-+-".join("-" * w for w in widths)

    print(fmt.format(*headers))
    print(sep)
    for row in rows:
        print(fmt.format(*[str(c) for c in row]))
    print()


def cap_name(allcaps: Dict[str, dict], uri: str) -> str:
    c = allcaps.get(uri)
    return c.get("name") if c else uri


def human_size(n: Any) -> str:
    try:
        x = int(n)
    except Exception:
        return str(n)
    units = ["B", "KiB", "MiB", "GiB", "TiB", "PiB"]
    i = 0
    v = float(x)
    while v >= 1024.0 and i < len(units) - 1:
        v /= 1024.0
        i += 1
    if i == 0:
        return f"{int(v)} {units[i]}"
    return f"{v:.1f} {units[i]}"


def fmt_epoch(s: Any) -> str:
    """Convert epoch seconds string → UTC timestamp string."""
    try:
        e = int(str(s))
        return dt.datetime.fromtimestamp(e, tz=dt.UTC).strftime("%Y-%m-%d %H:%M:%SZ")
    except Exception:
        return str(s)


## 2) Configuration
Edit these variables (or use environment variables) to point at the right API and defaults.


In [ ]:
BASE_URL = os.environ.get("IRI_BASE_URL", "https://iri-dev.ppg.es.net/api/v1")

home_path = Path("~/.iri_token.json").expanduser()
TOKEN = os.environ.get("IRI_API_TOKEN")
if TOKEN:
    print(f"IMPORTANT: Token defined inside .env file (or as environment variable). WILL NOT LOAD TOKEN from {home_path}")
else:
    # Load token from home_path
    with open(home_path) as f:
        tokendict = json.load(f)
        TOKEN = tokendict.get("IRI_API_TOKEN")
# If the token is still not available. print warning!
if not TOKEN:
    print(f"WARNING! Was unable to receive token from env variable or {home_path}. Use login-globus or login-esnet notebook to receive a token.")
    print("WARNING! FURTHER COMMANDS WILL NOT SUCCEED.")

# Optional defaults for compute job
DEFAULT_JOB_DIR = os.environ.get("IRI_JOB_DIR", "/data/home/jbalcas")
DEFAULT_QUEUE = os.environ.get("IRI_QUEUE", "debug")
DEFAULT_ACCOUNT = os.environ.get("IRI_ACCOUNT", "interactive")

HEADERS = {"Authorization": f"Bearer {TOKEN}", "Accept": "application/json"}
POLL_INTERVAL = 5
TIMEOUT = 180


## 3) Filesystem async-task helpers
Submit and poll filesystem tasks (`mkdir`, `ls`, `download`, etc.).


In [ ]:
def submit_fs_task(method: str, path: str, **kwargs) -> dict:
    url = f"{BASE_URL}{path}"
    print(f"Submitting {method} {url}")
    r = requests.request(method, url, headers=HEADERS, timeout=TIMEOUT, **kwargs)
    if not r.ok:
        die(f"{method} {url} failed: {r.status_code} {r.text}")

    data = r.json()
    if not data.get("task_id"):
        die(f"No task_id in response: {data}")
    if not data.get("task_uri"):
        die(f"No task_uri in response: {data}")
    return data


def wait_fs_task(taskin: dict) -> Any:
    deadline = time.time() + TIMEOUT
    while time.time() < deadline:
        r = requests.get(taskin["task_uri"], headers=HEADERS, timeout=TIMEOUT)
        if not r.ok:
            die(f"Task query failed: {r.status_code} {r.text}")

        t = r.json()
        status = t["status"]
        print(f"   FS Task {taskin['task_id']}: {status}")

        if status == "completed":
            return t.get("result")
        if status in ("failed", "canceled"):
            die(f"FS Task {taskin['task_id']} ended with status {status}: {t}")

        time.sleep(POLL_INTERVAL)

    die(f"FS Task {taskin['task_id']} timed out")


## 4) Discovery & resource selection
Lists projects/capabilities/resources and prompts you to pick **compute** and **filesystem** resources.


In [ ]:
def discover_resources_and_caps():
    print("\n" + "=" * 40)
    print("=== DISCOVERING RESOURCES AND CAPABILITIES ===")

    projects = requests.get(f"{BASE_URL}/account/projects", headers=HEADERS, timeout=TIMEOUT).json()
    project_rows = [[p.get("id"), p.get("name", ""), p.get("description", "")] for p in projects]
    print("\n=== PROJECTS ===")
    print_table(["Project ID", "Name", "Description"], project_rows)

    caps = requests.get(f"{BASE_URL}/account/capabilities", headers=HEADERS, timeout=TIMEOUT).json()
    cap_rows = [[c.get("self_uri"), c.get("name"), c.get("description", "")] for c in caps]
    cap_by_uri = {c["self_uri"]: c for c in caps}
    print("\n=== CAPABILITIES ===")
    print_table(["Capability URI", "Name", "Description"], cap_rows)

    alloc_rows = []
    project_caps = set()
    for pr in projects:
        allocs = requests.get(
            f"{BASE_URL}/account/projects/{pr['id']}/project_allocations",
            headers=HEADERS,
            timeout=TIMEOUT,
        ).json()
        for a in allocs:
            alloc_rows.append([pr["id"], a.get("id"), cap_name(cap_by_uri, a.get("capability_uri"))])
            if a.get("capability_uri") in cap_by_uri:
                project_caps.add(a["capability_uri"])

    print("\n=== PROJECT ALLOCATIONS ===")
    print_table(["Project ID", "Allocation ID", "Capability URI"], alloc_rows)

    if not project_caps:
        die("No allocations found for any project (need at least one capability).")

    resources = requests.get(
        f"{BASE_URL}/status/resources?offset=0&limit=100",
        headers=HEADERS,
        timeout=TIMEOUT,
    ).json()

    resource_rows = []
    for r in resources:
        rcaps = r.get("capability_uris", []) or []
        if any(cap in rcaps for cap in project_caps):
            cap_names = [cap_name(cap_by_uri, c) for c in rcaps]
            resource_rows.append(
                [
                    r.get("id"),
                    r.get("name", ""),
                    r.get("resource_type", ""),
                    r.get("description", ""),
                    ", ".join(cap_names),
                    r.get("current_status", ""),
                ]
            )

    print("\n=== RESOURCES (matching your project allocations) ===")
    print_table(
        ["Resource ID", "Name", "Type", "Description", "Capability URIs", "Current Status"],
        resource_rows,
    )

    return projects, cap_by_uri, resources, project_caps


def filter_resources(resources, cap_by_uri, project_caps, kind: str):
    kind = kind.lower().strip()
    filtered = []
    for r in resources:
        rcaps = r.get("capability_uris", []) or []
        if not any(cap in rcaps for cap in project_caps):
            continue

        rtype = (r.get("resource_type") or "").lower()
        capnames = " ".join((cap_name(cap_by_uri, c) or "").lower() for c in rcaps)

        if kind == "compute":
            if "compute" in rtype or "job" in rtype or "slurm" in rtype or "compute" in capnames:
                filtered.append(r)
        elif kind == "filesystem":
            if (
                "storage" in rtype
                or "filesystem" in rtype
                or rtype == "fs"
                or "storage" in capnames
                or "filesystem" in capnames
            ):
                filtered.append(r)
        else:
            filtered.append(r)
    return filtered


def prompt_pick_resource(resources, cap_by_uri, title: str) -> str:
    rows = []
    for r in resources:
        rcaps = r.get("capability_uris", []) or []
        capnames = ", ".join(cap_name(cap_by_uri, c) for c in rcaps)
        rows.append([r.get("id"), r.get("name", ""), r.get("resource_type", ""), capnames, r.get("current_status", "")])

    print("\n" + "=" * 40)
    print(f"=== {title} ===")
    print_table(["Resource ID", "Name", "Type", "Capabilities", "Status"], rows)

    rid = input(f"Enter {title.lower()} resource ID:\n").strip()
    if not rid:
        die("No resource ID provided.")
    return rid


## 5) Compute job submission & status polling functions
Functions to submit the compute job and poll its status until completion.


In [ ]:
def submit_compute_job(compute_resource_id: str, payload: dict) -> str:
    url = f"{BASE_URL}/compute/job/{compute_resource_id}"
    print("\n" + "=" * 40)
    print(f"=== SUBMIT COMPUTE JOB: {url} ===")

    r = requests.post(url, headers={**HEADERS, "Content-Type": "application/json"}, json=payload, timeout=TIMEOUT)
    if not r.ok:
        die(f"Compute submit failed: {r.status_code} {r.text}")

    data = r.json()
    job_id = data.get("id") or data.get("job_id")
    if not job_id:
        die(f"Compute submit response missing job id: {data}")

    print(f"Compute job submitted: job_id={job_id}")
    return job_id


def poll_compute_status(compute_resource_id: str, job_id: str) -> dict:
    url = f"{BASE_URL}/compute/status/{compute_resource_id}/{job_id}"
    print("\n" + "=" * 40)
    print(f"=== POLLING COMPUTE STATUS: {url} ===")

    deadline = time.time() + TIMEOUT
    last = {}
    while time.time() < deadline:
        r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        if not r.ok:
            die(f"Compute status failed: {r.status_code} {r.text}")

        last = r.json()
        st = (last.get("status") or {})
        state = (st.get("state") or "").lower()
        msg = st.get("message", "")
        exit_code = st.get("exit_code", None)

        print(f"   job {job_id}: state={state} message={msg} exit_code={exit_code}")

        if state in {"completed", "failed", "canceled", "cancelled", "timeout", "timed_out"}:
            return last

        time.sleep(POLL_INTERVAL)

    die(f"Compute job {job_id} timed out while polling status.")


## 6) Log retrieval helpers
Pretty `ls` output + base64 decode for `download`.


In [ ]:
def render_ls_table(ls_result: Any, base_path: str) -> None:
    if not isinstance(ls_result, dict):
        print("LS result (unexpected type):", ls_result)
        return

    items = ls_result.get("output")
    if not isinstance(items, list):
        print("LS result (unexpected shape):", ls_result)
        return

    def key(it):
        t = (it.get("type") or "").lower()
        is_dir = 0 if t == "directory" else 1
        return (is_dir, (it.get("name") or "").lower())

    items_sorted = sorted(items, key=key)

    rows = []
    for it in items_sorted:
        rows.append(
            [
                it.get("type", ""),
                it.get("permissions", ""),
                it.get("user", ""),
                it.get("group", ""),
                human_size(it.get("size", "")),
                fmt_epoch(it.get("last_modified", "")),
                it.get("name", ""),
                it.get("link_target") or "",
            ]
        )

    print(f"\nListing: {base_path}")
    print_table(["Type", "Perm", "UID", "GID", "Size", "Last Modified (UTC)", "Name", "Link Target"], rows)


def extract_base64_payload(download_result: Any) -> Optional[str]:
    if not isinstance(download_result, dict):
        return None

    out = download_result.get("output")

    if isinstance(out, str):
        return out

    if isinstance(out, dict):
        for k in ("content_base64", "base64", "data_base64", "data", "content"):
            v = out.get(k)
            if isinstance(v, str) and v.strip():
                return v
    return None


def decode_base64_to_text(b64: str) -> str:
    try:
        raw = base64.b64decode(b64, validate=True)
    except Exception:
        raw = base64.b64decode(b64)

    try:
        return raw.decode("utf-8")
    except UnicodeDecodeError:
        return raw.decode("utf-8", errors="replace")


## 7) Prepare for Smoke test
Identify available resources and select one for the compute and filesystem


In [ ]:
projects, cap_by_uri, resources, project_caps = discover_resources_and_caps()

compute_candidates = filter_resources(resources, cap_by_uri, project_caps, "compute")
fs_candidates = filter_resources(resources, cap_by_uri, project_caps, "filesystem")

if not compute_candidates:
    print("\n(No compute candidates matched heuristics; showing all matching resources instead.)")
    compute_candidates = [r for r in resources if any(c in (r.get("capability_uris") or []) for c in project_caps)]

if not fs_candidates:
    print("\n(No filesystem candidates matched heuristics; showing all matching resources instead.)")
    fs_candidates = [r for r in resources if any(c in (r.get("capability_uris") or []) for c in project_caps)]

COMPUTE_RESOURCE_ID = prompt_pick_resource(compute_candidates, cap_by_uri, "COMPUTE RESOURCE")
FS_RESOURCE_ID = prompt_pick_resource(fs_candidates, cap_by_uri, "FILESYSTEM RESOURCE")

print("\nChosen compute resource:", COMPUTE_RESOURCE_ID)
print("Chosen filesystem resource:", FS_RESOURCE_ID)

## 8) Submit compute job
Prepare bash script for job and submit job

In [ ]:
timestamp = dt.datetime.now(dt.UTC).strftime("%Y%m%d-%H%M%S")

job_log_name = f"esnet_iri_test_{timestamp}.log"
job_log_path = f"{DEFAULT_JOB_DIR.rstrip('/')}/{job_log_name}"

stdout_path = f"{DEFAULT_JOB_DIR.rstrip('/')}/esnet_iri_test_stdout_{timestamp}.log"
stderr_path = f"{DEFAULT_JOB_DIR.rstrip('/')}/esnet_iri_test_stderr_{timestamp}.log"

# ========================================================
# ========================================================
bash_payload = f"""
set -euo pipefail
echo "=== IRI compute smoke test ==="
echo "UTC now: $(date -u '+%Y-%m-%dT%H:%M:%SZ')"
echo "Hostname: $(hostname)"
echo "User: $(id || true)"
echo
echo "=== ENV (sorted) ==="
env | sort
echo
echo "=== Writing log file: {job_log_path} ==="
{{
  echo "IRI compute smoke test log"
  echo "UTC now: $(date -u '+%Y-%m-%dT%H:%M:%SZ')"
  echo "Hostname: $(hostname)"
  echo "---- ENV ----"
  env | sort
}} > "{job_log_path}"
echo "Wrote: {job_log_path}"
"""
# =======================================================
# =======================================================

compute_payload = {
    "executable": "bash",
    "arguments": ["-lc", bash_payload],
    "directory": DEFAULT_JOB_DIR,
    "name": f"iri-test-job-{timestamp}",
    "inherit_environment": True,
    "environment": {},
    "stdout_path": stdout_path,
    "stderr_path": stderr_path,
    "resources": {
        "node_count": 1,
        "process_count": 1,
        "processes_per_node": 1,
        "cpu_cores_per_process": 1,
        "exclusive_node_use": True,
        "memory": 268435456,
    },
    "attributes": {
        "duration": 600,
        "queue_name": DEFAULT_QUEUE,
        "account": DEFAULT_ACCOUNT,
        "custom_attributes": {},
    },
}

# ===========================================================
# SUBMIT JOB
job_id = submit_compute_job(COMPUTE_RESOURCE_ID, compute_payload)

## 9) Pool job status
Pool job status until job is finished

In [ ]:
final_status = poll_compute_status(COMPUTE_RESOURCE_ID, job_id)

state = ((final_status.get("status") or {}).get("state") or "").lower()
exit_code = (final_status.get("status") or {}).get("exit_code", None)

print("\n" + "=" * 40)
print("=== COMPUTE JOB FINISHED ===")
print(f"job_id   : {job_id}")
print(f"state    : {state}")
print(f"exit_code: {exit_code}")
print("=" * 40)

if state != "completed" or (exit_code not in (None, 0)):
    die(f"Compute job did not complete cleanly (state={state}, exit_code={exit_code}).")

## 10) List all files and download log file

In [ ]:
print("\n" + "=" * 40)
print("=== FILESYSTEM: LS JOB DIR ===")
task = submit_fs_task("GET", f"/filesystem/ls/{FS_RESOURCE_ID}", params={"path": DEFAULT_JOB_DIR})
ls_result = wait_fs_task(task)
render_ls_table(ls_result, DEFAULT_JOB_DIR)

print("\n" + "=" * 40)
print("=== FILESYSTEM: DOWNLOAD JOB LOG FILE ===")
task = submit_fs_task("GET", f"/filesystem/download/{FS_RESOURCE_ID}", params={"path": job_log_path})
download_result = wait_fs_task(task)

b64 = extract_base64_payload(download_result)
if not b64:
    die(f"Download result did not contain a base64 payload: {download_result}")

decoded = decode_base64_to_text(b64)

print("\n--- Decoded file content (UTF-8 best-effort) ---\n")
print(decoded)

local_out = f"downloaded_{job_log_name}"
with open(local_out, "w", encoding="utf-8") as f:
    f.write(decoded)